# From text to structured data

A resume is a wall of text. To compare candidates fairly you need fields:
name, skills, experience, education - the same shape for everyone. That is
what `extract_candidate` and `extract_job_description` do: one LLM call
each, constrained to a schema (`with_structured_output`), so what comes back
is always the same shape instead of free-form prose.

One thing worth opening `recruiting/extract.py` to notice: the schema has no
field for age, gender, marital status, nationality, or a photo. That is not
filtered out afterwards - it is never asked for in the first place.

## Step 1 - set up

You need a `LITELLM_API_KEY` in `.env` for this one (the iHQ LiteLLM proxy -
see `recruiting/llm.py`). Drive is optional today - we start from the sample
data in `sample_data/`, which works with zero Google setup.

In [ ]:
import os, getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("LITELLM_API_KEY"):
    os.environ["LITELLM_API_KEY"] = getpass.getpass("LiteLLM API key: ")

from recruiting import extract_candidate, extract_job_description

SAMPLE_DIR = Path("sample_data")
print("ready")

## Step 2 - extract the job description

The same call works on text fetched from Drive
(`drive.fetch_text(jd_file)` from notebook 00) - we use the sample file
here so this notebook runs on its own.

In [2]:
jd_text = (SAMPLE_DIR / "job_description.txt").read_text(encoding="utf-8")
jd = extract_job_description(jd_text, source_file="job_description.txt")

print(jd.title, "\n")
print("Essential:")
for r in jd.essential_requirements:
    print(" -", r)
print("\nPreferred:")
for r in jd.preferred_requirements:
    print(" -", r)

Backend Software Engineer 

Essential:
 - 3+ years of professional software engineering experience
 - Strong proficiency in Python
 - Experience designing and consuming REST APIs
 - Experience with relational databases (PostgreSQL, MySQL)
 - Bachelor's degree in Computer Science or related field

Preferred:
 - Experience with Docker and containerized deployments
 - Familiarity with cloud platforms (AWS, GCP, Azure)
 - Experience with asynchronous task queues (Celery, RQ)
 - Experience mentoring junior engineers


## Step 3 - extract one resume

In [3]:
resume_text = (SAMPLE_DIR / "resume_amina_hassan.txt").read_text(encoding="utf-8")
candidate = extract_candidate(resume_text, source_file="resume_amina_hassan.txt")

print(candidate.name, "|", candidate.email, "|", candidate.phone)
print("\nSkills:", ", ".join(candidate.skills))
print("\nExperience:")
for role in candidate.experience:
    print(" -", role)

Amina Hassan | amina.hassan.dev@example.com | +20 100 555 1234

Skills: Python, Django, Flask, REST API design, PostgreSQL, MySQL, Docker, AWS (ECS, RDS, S3), Celery, Git, code review, mentoring

Experience:
 - Backend Engineer, PayFlow Egypt (Cairo) — Jun 2021 to Present: Designed and shipped REST APIs (Django REST Framework) powering PayFlow's mobile app, serving ~40k daily active users. Owned the PostgreSQL schema for the transactions service; led a migration that cut p95 query latency by 60%. Containerized all backend services with Docker and deployed them on AWS (ECS, RDS, S3). Introduced Celery for async settlement processing, replacing a cron-based batch job. Mentored two junior engineers through their first six months.
 - Software Engineer, Cairo Byte Labs (Cairo) — Jul 2019 to May 2021: Built internal REST APIs in Python (Flask) for an inventory management product, backed by MySQL. Wrote the team's first integration test suite, raising coverage from 20% to 75%.


## What just happened

Two LLM calls, each constrained to a schema. The result is a plain
`Candidate` / `JobDescription` - printable, comparable, one step from JSON.
Everything from here on (matching, questions, bias checks) works on these
objects, not on raw text or on Drive directly.

Your turn: run `extract_candidate` on `resume_karim_elsayed.txt` and
`resume_lina_farouk.txt` too, and compare what came out. Karim's experience
is in a different stack than the JD asks for, and Lina's resume is vaguer
about dates and evidence than Amina's - keep both in mind, they come back in
notebook 02.